<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_judgments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review judgement

## Set up

In [1]:
# @title Dependencies

# Installations
!pip install -q -U transformers peft accelerate bitsandbytes huggingface_hub
!pip install -q pandas tqdm

# First-party imports
from google.colab import drive
import google.colab.userdata
import random
import json
import os
import math
import time

# Third-party imports
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from peft import PeftModel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 136.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 55.4 MB/s eta 0:00:00


In [2]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews_processed")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews_judged")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.jsonl")

# Define the full log path (final Parquet output)
LOG_PATH = os.path.join(RAW_DIR, "llm_reviews_log.parquet")
# Define the full log path for intermediate JSONL storage
LOG_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_log.jsonl")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_processed.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged.


In [3]:
# @title Setup definitions

### --- Settings --- ###

SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the model
MAX_CONCURRENT_CALLS = 20

# Subsetting dataset content
SUBSETTING_ACTIVE = False # SUBSETTING_ACTIVE = True activates the data subsetting
NUM_SUBSET = 40
RANDOM_SAMPLER = 42 # random

# Error probabilities for simulation mode
ERROR_PROB_LLM = 0.15 # random
ERROR_PROB_PARSING = 0.15 # random

# Define specific error strings from review generation/extraction/parsing failures
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"
UNKNOWN_ERROR_STATE_PARALLEL_PRCOESSING = "ERROR: UNKNOWN STATE REACHED IN PARALLEL PROCESSING"
EXTRACTION_FAILED_EMPTY_SECTION = "ERROR: EXTRACTION FAILED / SECTION MISSING"

# Define specific error strings from review judgement (and parsing) failures
JUDGE_GENERATION_FAILURE = "Error: Failed to generate judgement."
UNKNOWN_ERROR_STATE = "Error: Unknown client error."
PARSING_ERROR = "Error: Failed to parse LLM response as JSON."

# Column name for the segmented hash
SEGMENT_HASH_COLUMN = 'segment_hash'

### --- Content --- ####

TASK_DESCRIPTION_HEADER = '''###Task Description: You are an expert in evaluating peer review comments with respect to different aspects. These aspects are aimed to maximize the utilization of the review comments for the authors. The primary purpose of the review is to help/guide authors in improving their drafts. Keep this in mind while evaluating the review point. Whenever you encounter a borderline case, think: “Will this review point help authors improve their draft?”. There is no correlation between the aspect score and the length of the review point.'''

ASPECTS_NO_EXAMPLES = {
"actionability":
'''
**Actionability**

**Definition:** Measures the level of actionability in a review point. We evaluate actionability based on two criteria:

1. **Explicit vs. Implicit**:
   - **Explicit:** Actions or suggestions that are direct or apparent. Authors can directly identify modifications they should apply to their draft. Clarification questions should be treated as explicit statements if they give a direct action.
   - **Implicit:** Actions that need to be inferred from the comment. This includes missing parts that need to be added. Authors can deduce what needs to be done after reading the comment.

2. **Concrete vs. Vague**:
   - **Concrete:** Once the action is identified, the authors know exactly what needs to be done and how to apply the action.
   - **Vague:** After identifying the action, the authors still don’t know how to carry out this action.

**Importance:** It’s more important for actions to be concrete so that authors know how to apply them. It's also preferred for actions to be stated directly rather than inferred.

**Actionability Scale (1-5):**

1. **1: Unactionable**
   - **Definition:** The comment lacks meaningful information to help authors improve the paper. Authors do not know what they should do after reading the comment.

2. **2: Borderline Actionable**
   - **Definition:** The comment includes an implicitly stated action or an action that can be inferred. However, the action itself is vague and lacks detail on how to apply it.

3. **3: Somewhat Actionable**
   - **Definition:** The comment explicitly states an action but is vague on how to execute it.

4. **4: Mostly Actionable**
   - **Definition:** The comment implicitly states an action but concretely states how to implement the inferred action.

5. **5: Highly Actionable**
   - **Definition:** The comment contains an explicit action and concrete details on how to implement it. Authors know exactly how to apply it.
''',

"grounding_specificity": '''

**Grounding Specificity**

**Definition:** Measures how explicitly a review comment refers to a specific part of the paper and how clearly it identifies the issue with that part. This helps authors understand what needs revision and why. Grounding specificity has two key components:

1. **Grounding:** How well the authors can identify the specific part of the paper being addressed.
   - **Weak Grounding:** The author can make an educated guess but cannot precisely identify the referenced part.
   - **Full Grounding:** The author can accurately pinpoint the section, table, figure, or unique aspect being addressed. This can be achieved through:
     - Literal mentions of sections, tables, figures, etc.
     - Mentions of unique elements of the paper.
     - General comments that clearly imply the relevant parts without explicitly naming them.

2. **Specificity:** How clearly the comment details what is wrong or missing in the referenced part. If external work is mentioned, it also evaluates whether specific examples are provided.

**Importance:** It's more important for the comment to be grounded than to be specific.

**Grounding Specificity Scale (1-5):**

1. **Not Grounded**
   - **Definition**: This comment is not grounded at all. It does not identify a specific area in the paper. The comment is highly unspecific.

2. **Weakly Grounded and Not Specific**
   - **Definition**: The authors cannot confidently determine which part the comment addresses. Further, the comment does not specify what needs to be addressed in this part.

3. **Weakly Grounded and Specific**
   - **Definition**: The authors cannot confidently determine which part the comment addresses. However, the comment clearly specifies what needs to be addressed in this part.

4. **Fully Grounded and Under-Specific**
   - **Definition**: The comment explicitly mentions which part of the paper it addresses, or it should be obvious to the authors. However, this comment does not specify what needs to be addressed in this part.

5. **Fully Grounded and Specific**
   - **Definition**: The comment explicitly mentions which part of the paper it addresses, and it is obvious to the authors. The comment specifies what needs to be addressed in this part.
''',

"verifiability":
'''
**Verifiability**

**Definition:** Evaluates whether a review comment contains a claim and, if so, how well that claim is supported using logical reasoning, common knowledge, or external references.

### **Step 1: Claim Extraction**

**Objective:**
Determine whether the given text contains a claim (i.e., an opinion, judgment, or suggestion) or consists solely of factual statements that require no verification.

**Claim Definition:**
A statement is considered a claim if it falls into one or more of the following categories:
- **Subjective opinions or disagreements** (e.g., criticism of an experimental choice).
- **Suggestions or requests for changes** (e.g., recommending removal, addition, or discussion).
- **Judgments about the paper** (e.g., stating something is unclear, not well-written, or lacks detail).
- **Deductions or inferred observations** that go beyond merely stating facts.
- **Statements requiring justification** to be understood or accepted.


**Normal Statements ("No Claim")**
A statement is classified as "X" if it:
- Describes facts without suggesting changes.
- Makes general statements about the paper without an opinion.
- Presents objective, verifiable facts that require no justification.
- Asks for clarifications or general questions.
- States logical statements or directly inferable information.
- Makes positive claims (e.g., *“The paper is well-written”*), as these do not help improve the work.

---

### **Step 2: Verifiability Verification**

**Objective:**
Assess how well a claim is verified by examining the reasoning, common knowledge, or external references provided. The purpose is to ensure that the review comment helps the authors improve their work.

**Verification Methods:**
A claim is considered verifiable if supported by one or more of the following:
- **Logical reasoning** – A clear explanation of why the claim is valid.
- **Common knowledge** – Reference to well-accepted practices or standards.
- **External references** – Citation of relevant literature, data, or sources.

**Verifiability Scale (1–5 & X):**

1. **1: Unverifiable**
   - The comment contains a claim without any supporting evidence or justification.
2. **2: Borderline Verifiable**
   - Some support is provided, but it is vague, insufficient, or difficult to follow.
3. **3: Somewhat Verifiable**
   - The claim has some justification but lacks key elements (e.g., examples, references).
4. **4: Mostly Verifiable**
   - The claim is well-supported but has minor gaps in explanation or references.
5. **5: Fully Verifiable**
   - The claim is thoroughly supported by explicit, sufficient, and robust evidence, such as:
     - Clear reasoning and precise explanations.
     - Specific references to external works.
     - Logical and unassailable common-sense arguments.
6. **X: No Claim**
- The comment contains only factual, descriptive statements without claims, opinions, or suggestions.

---

### **Instructions for Evaluation:**
1. **Extract Claims:** Determine whether the review comment contains a claim or is a normal statement. If there is no claim, score it as "X"
2. **Assess Verifiability:** If a claim exists, score it based on how well it is justified from 1 to 5.
'''
,

"helpfulness" : '''
**Helpfulness**

**Definition:** Assign a subjective score to reflect the value of the review comment to the authors. Helpfulness is rated on a scale from 1 to 5, with the following definitions:

1. **1: Not Helpful at All**
   - **Definition:** The comment fails to identify meaningful weaknesses or suggest improvements, leaving the authors with no actionable feedback.

2. **2: Barely Helpful**
   - **Definition:** The comment identifies a weakness or improvement area but is vague, lacks clarity, or provides minimal guidance, making it only slightly beneficial for the authors.

3. **3: Somewhat Helpful**
   - **Definition:** The comment identifies weaknesses or areas for improvement but is incomplete or lacks depth. While the authors gain some insights, the feedback does not fully address their needs for improving the draft.

4. **4: Mostly Helpful**
   - **Definition:** The comment provides clear and actionable feedback on weaknesses and areas for improvement, though it could be expanded or refined to be fully comprehensive and impactful.

5. **5: Highly Helpful**
   - **Definition:** The comment thoroughly identifies weaknesses and offers detailed, actionable, and constructive suggestions that empower the authors to significantly improve their draft.
'''
}

INSTRUCTION_SCORE_ONLY_PROMPT_TAIL = '''
###Instruction:
Evaluate the review based on the given definitions of the aspect(s) above. Output only the score. The possbile values for scores are 1-5 and X.

Generate the output in JSON format with the following format:

   "actionability_label": "",
   "grounding_specificity_label": "",
   "verifiability_label": "",
   "helpfulness_label": ""



###Review Point: [REVIEW POINT]
###Output: {
  "actionability_label": "[ACTIONABILITY LABEL]",
  "grounding_specificity_label": "[GROUNDING SPECIFICITY LABEL]",
  "verifiability_label": "[VERIFIABILITY LABEL]",
  "helpfulness_label": "[HELPFULNESS LABEL]"
};'''

### --- API/Client --- ###

# HF token key
hf_token = google.colab.userdata.get('HF_TOKEN')
login(token=hf_token)

MAX_RETRIES = 3
RETRY_DELAY = 3


In [4]:
# @title Install fine-tuned model

# Setup names and config
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_MODEL = "boda/RevUtil_Llama-3.1-8B-Instruct_score_only"

# Conditionally load model and tokenizer based on SIMULATION_ACTIVE
if not SIMULATION_ACTIVE:

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)

    # Set pad token
    tokenizer.pad_token = tokenizer.eos_token

    # Configure 4-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # Load Base model
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quantization_config,
        device_map="auto",
        attn_implementation="sdpa",
        token=hf_token,
        trust_remote_code=True,
    )

    # Attach the RevUtil Adapter
    model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_MODEL,
        token=hf_token
    )

    # Set model to eval mode
    model.eval()

    # Configure model with pad token
    model.config.pad_token_id = tokenizer.pad_token_id
    tokenizer.padding_side = "left"

else:
    print("SIMULATION_ACTIVE is True. Skipping model and tokenizer loading to save resources.")
    tokenizer = None
    model = None

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

In [5]:
# @title Data import

# Read processed llm reviews
df_reviews = pd.read_parquet(os.path.join(DATASET_DIR, "llm_reviews_segmented.parquet"))

# Check df_reviews
display(df_reviews.shape)
display(df_reviews.head(3))


(1935, 17)

,segment_hash,config_group,paper_id,run_signature,model,temperature,reasoning_effort,chain_of_thought,note_taking,llm_notes,llm_review,extracted_weaknesses,extracted_scores,segmented_comment,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating
0,9a59cd16563e,model: gpt-5-mini-2025-08-07 | temperature: na...,vuD2xEtxZcj,44938bfcb50e,gpt-5-mini-2025-08-07,NaN,low,,Faithful,This is a simulated note based on: Model='gpt-...,"{""summary_of_paper"": ""Simulated summary for gp...",[Simulated weakness 1 for gpt-5-mini-2025-08-0...,"{'correctness_rating': 1.0, 'empirical_novelty...",Simulated weakness 1 for gpt-5-mini-2025-08-07,1.0,3.0,3.0
1,2270c9e4126f,model: gpt-5-mini-2025-08-07 | temperature: na...,vuD2xEtxZcj,44938bfcb50e,gpt-5-mini-2025-08-07,NaN,low,,Faithful,This is a simulated note based on: Model='gpt-...,"{""summary_of_paper"": ""Simulated summary for gp...",[Simulated weakness 1 for gpt-5-mini-2025-08-0...,"{'correctness_rating': 1.0, 'empirical_novelty...",Simulated weakness 2 for gpt-5-mini-2025-08-07,1.0,3.0,3.0
2,9e246997d09a,model: gpt-5-mini-2025-08-07 | temperature: na...,WoByU5W5te0,44938bfcb50e,gpt-5-mini-2025-08-07,NaN,low,,Faithful,ERROR: NOTE TAKING NOT SUCCESSFUL,ERROR: REVIEW GENERATION NOT SUCCESSFUL,[ERROR: REVIEW GENERATION NOT SUCCESSFUL],"{'correctness_rating': None, 'empirical_novelt...",ERROR: REVIEW GENERATION NOT SUCCESSFUL,NaN,NaN,NaN


In [6]:
# @title Subset data (if active)

if SUBSETTING_ACTIVE:
    df_reviews = df_reviews.sample(n=NUM_SUBSET, random_state=RANDOM_SAMPLER)
    print(f"Data randomly subsetted to {NUM_SUBSET} rows.")

    display(df_reviews.shape)
    display(df_reviews.head(3))
else:
    print("Paper content subsetting is inactive.")

Data randomly subsetted to 40 rows.


(40, 17)

,segment_hash,config_group,paper_id,run_signature,model,temperature,reasoning_effort,chain_of_thought,note_taking,llm_notes,llm_review,extracted_weaknesses,extracted_scores,segmented_comment,correctness_rating,technical_novelty_significance_rating,empirical_novelty_significance_rating
582,71911c54fd0b,model: gpt-4o-2024-11-20 | temperature: 1.5 | ...,QVcDQJdFTG,44938bfcb50e,gpt-4o-2024-11-20,1.5,None,Explain your thought process step by step.,Turned off,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""Simulated summary for gp...","[Simulated weakness 1 for gpt-4o-2024-11-20, S...","{'correctness_rating': 4.0, 'empirical_novelty...",Simulated weakness 1 for gpt-4o-2024-11-20,4.0,5.0,4.0
901,1db587cd84f9,model: o4-mini-2025-04-16 | temperature: nan |...,yHIIM9BgOo,44938bfcb50e,o4-mini-2025-04-16,NaN,low,,Faithful,This is a simulated note based on: Model='o4-m...,"{""summary_of_paper"": ""Simulated summary for o4...","[Simulated weakness 1 for o4-mini-2025-04-16, ...","{'correctness_rating': 2.0, 'empirical_novelty...",Simulated weakness 2 for o4-mini-2025-04-16,2.0,2.0,2.0
907,e004515fd6bb,model: o4-mini-2025-04-16 | temperature: nan |...,WoByU5W5te0,44938bfcb50e,o4-mini-2025-04-16,NaN,low,,Turned off,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""Simulated summary for o4...","[Simulated weakness 1 for o4-mini-2025-04-16, ...","{'correctness_rating': 3.0, 'empirical_novelty...",Simulated weakness 2 for o4-mini-2025-04-16,3.0,3.0,2.0


## Define

In [7]:
# @title Prompt construction

def build_prompt(review_point: str) -> str:
    """Constructs the full prompt as a single string for the client message."""
    # Initialize prompt
    content = []
    # Append task description
    content.append(TASK_DESCRIPTION_HEADER)

    # Format keys and append their definitions
    for aspect_key, aspect_definition_text in ASPECTS_NO_EXAMPLES.items():
        if aspect_key == "grounding_specificity":
            display_aspect_name = "Grounding & Specificity"
        else:
            display_aspect_name = aspect_key.replace('_', ' ').title()
        content.append(f"Aspect: {display_aspect_name}\n{aspect_definition_text}")

    # Replace placeholder with actual review segment and append
    formatted_instruction_part = INSTRUCTION_SCORE_ONLY_PROMPT_TAIL.replace("[REVIEW POINT]", review_point)
    content.append(formatted_instruction_part)

    # Join all appended prompts parts to a single string
    full_prompt_string = "\n".join(content)

    return full_prompt_string

In [8]:
# @title Client

class ScoringClient:
    def __init__(self, local_model, local_tokenizer, simulate: bool = False, seed: int = 42):
        """
        Initialize ScoringClient for local model inference.
        This client handles both simulation and actual local inference.
        """
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.local_model = local_model
        self.local_tokenizer = local_tokenizer

        if not self.simulate:
            print(f"ScoringClient initialized for local model.")
        else:
            print("ScoringClient running in simulation mode.")

    def score_batch_review_points(self, review_items: list[dict]) -> list[str]:
        """
        Scores a batch of review items, either by simulating responses or by calling
        the local LLM for batched inference.
        Returns a list of raw output strings from the LLM or simulation, one for each item.
        """
        prompts = [build_prompt(item['segmented_comment']) for item in review_items]
        raw_outputs = []

        if self.simulate:
            for item in review_items:
                random_val = self.rng.random()
                # Simulate failure in the judgement process
                if random_val < ERROR_PROB_LLM:
                    raw_outputs.append(JUDGE_GENERATION_FAILURE)
                # Simulate failure in the parsing process by returning an malformed JSON string
                elif random_val < ERROR_PROB_LLM + ERROR_PROB_PARSING:
                    raw_outputs.append("{'actionability_label': 'invalid' 'grounding_specificity_label': ")
                # Simulate artificial but valid string
                else:
                    simulated_output_dict = {
                        "actionability_label": str(self.rng.randint(1, 5)),
                        "grounding_specificity_label": str(self.rng.randint(1, 5)),
                        "verifiability_label": self.rng.choice([str(self.rng.randint(1, 5)), "X"]),
                        "helpfulness_label": str(self.rng.randint(1, 5))
                    }
                    raw_outputs.append(json.dumps(simulated_output_dict))
        else:
            print(f"[{time.strftime('%H:%M:%S')}] Starting tokenization for batch of {len(prompts)} prompts...")
            # Tokenize the batch of prompts
            inputs = self.local_tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(self.local_model.device)
            print(f"[{time.strftime('%H:%M:%S')}] Tokenization complete. Starting model generation...")

            # Call model for a batched generation
            with torch.no_grad():
                outputs = self.local_model.generate(
                    input_ids=inputs["input_ids"], # Start token
                    attention_mask=inputs["attention_mask"], # General token setting
                    max_new_tokens=256, # Sufficient to output the complete expected JSON response schema
                    num_return_sequences=1, # Amount of responses to generate per review comment
                    pad_token_id=self.local_tokenizer.pad_token_id, # Optimize tokenization for parallel processing
                    eos_token_id=self.local_tokenizer.eos_token_id, # End-of-sequence token
                    do_sample=False, # Choose the most likely next token
                    use_cache=True, # Stores intermediate token information
                )
            print(f"[{time.strftime('%H:%M:%S')}] Model generation complete. Starting decoding...")

            # Decode each generated sequence in the batch
            for i in range(len(prompts)):
                # Get the actual unpadded length of the i-th input prompt
                input_length = inputs["attention_mask"][i].sum().item()
                # Slice new tokens (actual response)
                generated_text = self.local_tokenizer.decode(outputs[i, input_length:], skip_special_tokens=True)

                # If response is empty (no new tokens)
                if not generated_text.strip():
                    raw_outputs.append(JUDGE_GENERATION_FAILURE)
                else:
                    raw_outputs.append(generated_text)
            print(f"[{time.strftime('%H:%M:%S')}] Decoding complete for batch.")
        return raw_outputs

## Run

In [ ]:
# @title Judgements

# Initialize client
client = ScoringClient(
    local_model=model if not SIMULATION_ACTIVE else None,
    local_tokenizer=tokenizer if not SIMULATION_ACTIVE else None,
    simulate=SIMULATION_ACTIVE
)

# --- Populate list of already processed segmented_comments (indicated by their hashes) ---

# Initialize list of hashes
processed_hashes = []

# Check if result file exists and has content
if os.path.exists(RESULTS_JSON_PATH):
    try:
        existing_results_df = pd.read_json(RESULTS_JSON_PATH, lines=True)
        if not existing_results_df.empty and SEGMENT_HASH_COLUMN in existing_results_df.columns:
            processed_hashes.extend(existing_results_df[SEGMENT_HASH_COLUMN].tolist())
    except Exception as e:
        print(f"Warning: Could not read existing results JSONL for IDs: {e}")

# Check if log file exists and has content
if os.path.exists(LOG_JSON_PATH) and os.path.getsize(LOG_JSON_PATH) > 0:
    try:
        existing_log_df = pd.read_json(LOG_JSON_PATH, lines=True)
        if not existing_log_df.empty and SEGMENT_HASH_COLUMN in existing_log_df.columns:
            processed_hashes.extend(existing_log_df[SEGMENT_HASH_COLUMN].tolist())
    except Exception as e:
        print(f"Warning: Could not read existing log JSONL for IDs: {e}")

# Get unique hash's and convert to a list
processed_hashes = list(set(processed_hashes))

print(f"Loaded {len(processed_hashes)} unique segmented_comment_ids from previous runs.")

# --- Start scoring of segmented_comments ---

def main():
    # Filter out already processed items
    df_reviews_to_process = df_reviews[~df_reviews[SEGMENT_HASH_COLUMN].isin(processed_hashes)]

    # If no review points to process
    if df_reviews_to_process.empty:
        print("No new review points to process.")
        return

    # Else count the still to be processed amount of batches
    total_reviews_to_process = len(df_reviews_to_process)
    processing_chunk_size = MAX_CONCURRENT_CALLS
    num_processing_chunks = math.ceil(total_reviews_to_process / processing_chunk_size)

    with tqdm(total=total_reviews_to_process, desc="Overall Scoring Progress") as pbar:
        # Iterate over batches
        for i in range(num_processing_chunks):
            start_idx = i * processing_chunk_size
            end_idx = min((i + 1) * processing_chunk_size, total_reviews_to_process)
            current_chunk_df = df_reviews_to_process.iloc[start_idx:end_idx]

            # Information about current state of execution
            print(f"\nProcessing batch {i + 1}/{num_processing_chunks} (items {start_idx} to {end_idx-1})...")

            # Convert current_chunk_df rows to a list of dicts for processing
            current_chunk_data = current_chunk_df.to_dict(orient='records')

            # Initialize lists to separate to be judged and already failed review comments
            items_to_score = []
            skipped_results = []

            for item in current_chunk_data:
                segment_comment = item['segmented_comment']
                if segment_comment in [REVIEW_GENERATION_FAILURE, UNKNOWN_ERROR_STATE_PARALLEL_PRCOESSING, EXTRACTION_FAILED_EMPTY_SECTION]:
                    # This item is an error, so skip scoring and record the error directly
                    result_entry = {
                        'paper_id': item['paper_id'],
                        'model': item['model'],
                        'temperature': item['temperature'],
                        'reasoning_effort': item['reasoning_effort'],
                        'chain_of_thought': item['chain_of_thought'],
                        'note_taking': item['note_taking'],
                        'segmented_comment': segment_comment,
                        SEGMENT_HASH_COLUMN: item[SEGMENT_HASH_COLUMN],
                        'config_group': item['config_group'],
                        'llm_raw_response': segment_comment
                    }
                    skipped_results.append(result_entry)
                    print(f"[Skipped]: paper_id {item['paper_id']} | segmented hash {item[SEGMENT_HASH_COLUMN]} - Already a generation failure.")
                else:
                    # Add item to the list as still to be judged
                    items_to_score.append(item)

            # Call the batch scoring function only for items that need judging
            raw_output_strings = []
            if items_to_score:
                raw_output_strings = client.score_batch_review_points(items_to_score)

            # Initialize result dict for processed (judged) items
            processed_scored_results = []
            # Populate result dict for each review comment that was judged in the batch
            for j, current_review_item in enumerate(items_to_score):
                print(f"[Model call]: paper_id {current_review_item['paper_id']} | segmented hash {current_review_item[SEGMENT_HASH_COLUMN]}")
                result_entry = {
                    'paper_id': current_review_item['paper_id'],
                    'model': current_review_item['model'],
                    'temperature': current_review_item['temperature'],
                    'reasoning_effort': current_review_item['reasoning_effort'],
                    'chain_of_thought': current_review_item['chain_of_thought'],
                    'note_taking': current_review_item['note_taking'],
                    'segmented_comment': current_review_item['segmented_comment'],
                    SEGMENT_HASH_COLUMN: current_review_item[SEGMENT_HASH_COLUMN],
                    'config_group': current_review_item['config_group'],
                    'llm_raw_response': raw_output_strings[j]
                }
                processed_scored_results.append(result_entry)

            # Combine skipped and judged results
            processed_results = skipped_results + processed_scored_results

            # Write results and logs for the current chunk to JSONL files in append mode
            with open(RESULTS_JSON_PATH, 'a') as results_jsonl_file:
                with open(LOG_JSON_PATH, 'a') as log_jsonl_file:
                    for result_entry in processed_results:
                        # Prepare log entry
                        log_entry = {
                            'paper_id': result_entry['paper_id'],
                            SEGMENT_HASH_COLUMN: result_entry[SEGMENT_HASH_COLUMN],
                            'segmented_comment': result_entry['segmented_comment'],
                        }

                        results_jsonl_file.write(json.dumps(result_entry) + '\n')
                        log_jsonl_file.write(json.dumps(log_entry) + '\n')

            # Update progress bar for the processed chunk
            pbar.update(len(current_chunk_data))

    print(f"All review points processed. Results saved to {RESULTS_JSON_PATH} and logs to {LOG_JSON_PATH}.")

# Run the main function
if __name__ == '__main__':
    main()

## Transformation

In [10]:
# @title Conversion

# Convert RESULTS_JSON_PATH to RESULTS_PATH (Parquet)
if os.path.exists(RESULTS_JSON_PATH):
    try:
        print(f"Converting {RESULTS_JSON_PATH} to {RESULTS_PATH}...")
        jsonl_df = pd.read_json(RESULTS_JSON_PATH, lines=True)
        if not jsonl_df.empty:
            jsonl_df.to_parquet(RESULTS_PATH, index=False)
            print(f"Successfully converted and saved results to {RESULTS_PATH}.")
        else:
            print(f"{RESULTS_JSON_PATH} is empty. No Parquet file generated for results.")
    except Exception as e:
        print(f"Error converting results JSONL to Parquet: {e}")
else:
    print(f"Intermediate results file '{RESULTS_JSON_PATH}' does not exist. No results Parquet file generated.")

# Convert LOG_JSON_PATH to LOG_PATH (Parquet)
if os.path.exists(LOG_JSON_PATH):
    try:
        print(f"Converting {LOG_JSON_PATH} to {LOG_PATH}...")
        jsonl_log_df = pd.read_json(LOG_JSON_PATH, lines=True)
        if not jsonl_log_df.empty:
            jsonl_log_df.to_parquet(LOG_PATH, index=False)
            print(f"Successfully converted and saved log to {LOG_PATH}.")
        else:
            print(f"'{LOG_JSON_PATH}' is empty. No Parquet file generated for log.")
    except Exception as e:
        print(f"Error converting log JSONL to Parquet: {e}")
else:
    print(f"Intermediate log file '{LOG_JSON_PATH}' does not exist. No log Parquet file generated.")

Converting /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.jsonl to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.parquet...
Successfully converted and saved results to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.parquet.
Converting /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.jsonl to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.parquet...
Successfully converted and saved log to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.parquet.


In [11]:
# @title Results

# Load result file
if os.path.exists(RESULTS_PATH):
    try:
        llm_results_df = pd.read_parquet(RESULTS_PATH)
        # Check results only if loading was successful
        display(llm_results_df)
        display(llm_results_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing results parquet file {RESULTS_PATH}. Error: {e}")
else:
    print(f"Results file '{RESULTS_PATH}' does not exist.")

# Load log file
if os.path.exists(LOG_PATH):
    try:
        llm_log_df = pd.read_parquet(LOG_PATH)
        # Check log only if loading was successful
        display(llm_log_df.head(5))
        display(llm_log_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing log parquet file {LOG_PATH}. Error: {e}")
else:
    print(f"Log file '{LOG_PATH}' does not exist.")

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,segmented_comment,segment_hash,config_group,llm_raw_response
0,XZRmNjUMj0c,gpt-4o-mini-2024-07-18,1.0,None,Explain your thought process step by step.,Faithful,ERROR: REVIEW GENERATION NOT SUCCESSFUL,ae395bf6efa4,model: gpt-4o-mini-2024-07-18 | temperature: 1...,ERROR: REVIEW GENERATION NOT SUCCESSFUL
1,QVcDQJdFTG,gpt-4o-2024-11-20,1.5,None,Explain your thought process step by step.,Turned off,Simulated weakness 1 for gpt-4o-2024-11-20,71911c54fd0b,model: gpt-4o-2024-11-20 | temperature: 1.5 | ...,"}; {\n ""actionability_label"": ""1"",\n ""gr..."
2,yHIIM9BgOo,o4-mini-2025-04-16,NaN,low,,Faithful,Simulated weakness 2 for o4-mini-2025-04-16,1db587cd84f9,model: o4-mini-2025-04-16 | temperature: nan |...,"LABEL]""\n}; ###Actionability Label: 1\n###Gro..."
3,WoByU5W5te0,o4-mini-2025-04-16,NaN,low,,Turned off,Simulated weakness 2 for o4-mini-2025-04-16,e004515fd6bb,model: o4-mini-2025-04-16 | temperature: nan |...,"LABEL]""\n}; ###Actionability Label: 1\n###Gro..."
4,vuD2xEtxZcj,llama-3.3-70b-versatile,1.5,None,Explain your thought process step by step.,Faithful,Simulated weakness 2 for llama-3.3-70b-versatile,a4a6b5711c57,model: llama-3.3-70b-versatile | temperature: ...,"LABEL]""\n}; {\n ""actionability_label"": ""1""..."
5,QVcDQJdFTG,gpt-3.5-turbo-0125,1.5,None,Explain your thought process step by step.,Faithful,Simulated weakness 1 for gpt-3.5-turbo-0125,f6e342e7fbe3,model: gpt-3.5-turbo-0125 | temperature: 1.5 |...,"}; {\n ""actionability_label"": ""1"",\n ""gr..."
6,Sh97TNO5YY_,gpt-3.5-turbo-0125,0.0,None,Explain your thought process step by step.,Faithful,Simulated weakness 2 for gpt-3.5-turbo-0125,d92ac07c4e3c,model: gpt-3.5-turbo-0125 | temperature: 0.0 |...,"}; {\n ""actionability_label"": ""1"",\n ""gr..."
7,deit1AdsFU,o3-mini-2025-01-31,NaN,high,Explain your thought process step by step.,Turned off,Simulated weakness 2 for o3-mini-2025-01-31,859709b7fe11,model: o3-mini-2025-01-31 | temperature: nan |...,"LABEL]""\n}; ###ACTIONABILITY_LABEL: ""1""\n###G..."
8,cDYRS5iZ16f,gpt-4o-mini-2024-07-18,1.5,None,Explain your thought process step by step.,Faithful,Simulated weakness 2 for gpt-4o-mini-2024-07-18,4ae2ef90cfaf,model: gpt-4o-mini-2024-07-18 | temperature: 1...,"{\n ""actionability_label"": ""1"",\n ""grou..."
9,XZRmNjUMj0c,gpt-3.5-turbo-0125,0.0,None,Explain your thought process step by step.,Faithful,Simulated weakness 1 for gpt-3.5-turbo-0125,99c59f43fc0a,model: gpt-3.5-turbo-0125 | temperature: 0.0 |...,"}; {\n ""actionability_label"": ""1"",\n ""gr..."


(40, 10)

,paper_id,segment_hash,segmented_comment
0,XZRmNjUMj0c,ae395bf6efa4,ERROR: REVIEW GENERATION NOT SUCCESSFUL
1,QVcDQJdFTG,71911c54fd0b,Simulated weakness 1 for gpt-4o-2024-11-20
2,yHIIM9BgOo,1db587cd84f9,Simulated weakness 2 for o4-mini-2025-04-16
3,WoByU5W5te0,e004515fd6bb,Simulated weakness 2 for o4-mini-2025-04-16
4,vuD2xEtxZcj,a4a6b5711c57,Simulated weakness 2 for llama-3.3-70b-versatile


(40, 3)

## Close

In [12]:
# @title Ensure runtime disconnection

from google.colab import runtime
runtime.unassign()